In [1]:
import pandas as pd
import numpy as np

In [2]:
# Reading mnist_784.csv dataset.
df = pd.read_csv('mnist_784.csv')

In [3]:
# Hyper parameters.
neural_network = [784, 128, 128, 128, 128, 10]
input_size = neural_network[0]
learning_rate = 0.001
epochs = 40
batch_size = 100
training_set_percentage = 0.70
test_set_percentage = 1 - training_set_percentage

In [4]:
# Inputs.
X = df.iloc[:,0:784].values/255
# Outputs.
Y0 = df.iloc[:,784].values

# One hot-encoding.
Y=np.zeros((70000,10)) 
for i in range(70000): 
    Y[i,Y0[i]]=1

# Training sets. 
training_size = int(len(X) * training_set_percentage)
X_training_set = X[:training_size, :].T
Y_training_set = Y[:training_size, :].T

# Test sets.
X_test_set = X[training_size:, :].T
Y_test_set = Y[training_size:, :].T

In [5]:
# Initialize neural network weights and biases (model parameters).
weights = []
biases = []

# Initialize weights with small random values.
for current_in, next_out in zip(neural_network, neural_network[1:]):
    weights.append(np.random.randn(next_out, current_in))

# Initialize biases with zeros.
for layer_size in neural_network[1:]:
    biases.append(np.zeros((layer_size, 1)))

In [8]:
# Training.
for epoch in range(epochs):

    error = 0
    num_batches = X_training_set.shape[1] // batch_size   # 490 batches
    
    for batch in range(num_batches):
        start = batch * batch_size
        end = (batch+1) * batch_size
        X_batch_training_set=X_training_set[:,start:end]
        Y_batch_training_set=Y_training_set[:,start:end]

        # Forward pass over the whole batch at once.

        # Initialize input matrix: each column is one image.
        input_matrix = X_batch_training_set

        # Initialize label matrix: each column is one one-hot label.
        actual_label = Y_batch_training_set
        
        # Forward propagation.
        # Start with the input_matrix.
        activated_neuron = input_matrix
        activated_neurons = []
        activated_neurons.append(activated_neuron)

        # Iterate through the layers.
        for layer in range(len(neural_network)-1):
            activated_neuron = 1 / (1 + np.exp(-((weights[layer] @ activated_neuron) + biases[layer])))
            activated_neurons.append(activated_neuron)

        # Last layer is the output.
        output_neurons = activated_neuron

        # Calculating the error using the mean absolute error loss function.
        error += np.sum(np.square(actual_label - output_neurons))
        
        # Backpropagation.
        
        # Calculate the delta.
        l_wr_o = 2 * (output_neurons - actual_label)
        o_wr_z = output_neurons * (1 - output_neurons)
        delta = l_wr_o * o_wr_z

        # Go backwards through layers.
        for layer in range(len(weights) - 1, -1, -1):

            # With respect to weight.
            z_wr_weight = activated_neurons[layer]
            # With respect to bias.
            z_wr_bias = np.ones((1, batch_size))

            # Gradients.
            l_wr_weight = (delta @ z_wr_weight.T) / batch_size
            l_wr_bias = (delta @ z_wr_bias.T) / batch_size

            # With respect to previous neuron.
            z_wr_q = weights[layer]
            # Next delta.
            next_delta = z_wr_q.T @ delta
            
            # Gradient descent.
            weights[layer] -= learning_rate * l_wr_weight
            biases[layer] -= learning_rate * l_wr_bias

            delta = next_delta
            
    print(error)

/tmp/ipykernel_26557/3078402833.py:29: RuntimeWarning: overflow encountered in exp
  activated_neuron = 1 / (1 + np.exp(-((weights[layer] @ activated_neuron) + biases[layer])))


17656.349702437426
17359.204181851816
17375.850413601645
17397.451408972363
17415.16061266184
17424.6502996954
17440.843421139074
17478.88947369271
17445.82439854922
17454.375569642485
17450.181378267145
17487.46451962697
17516.39332198382
17551.58144272002
17584.24835534786
17620.305537923465
17679.938563672193
17696.695757264395
17713.284033499905
17763.596508686616
17778.71071506513
17816.391035598688
17851.24689148288
17909.31754210606
17918.443566275757
17910.544230271444
17930.41666012173
17998.346675867728
18017.07304108604
18027.40411812989
18055.87224672803
18064.154548784
18107.943917853343
18162.17032465677
18191.53464297457
18226.394201798397
18242.676783362447
18242.4880415376
18260.349987297363
18277.929450220574


In [9]:
correct = 0

# Starting neuron.
activated_neuron = X_test_set

# Iterate through the layers.
for layer in range(len(neural_network)-1):
    activated_neuron = 1 / (1 + np.exp(-((weights[layer] @ activated_neuron) + biases[layer])))

# Last layer is the output.
output_neurons = activated_neuron

# Predicted and actual digit for every test image (down each column).
predicted_digits = np.argmax(output_neurons, axis=0)
actual_digits = np.argmax(Y_test_set, axis=0)

# Count the matches.
correct = np.sum(predicted_digits == actual_digits)

# Calculate accuracy.
accuracy = correct / Y_test_set.shape[1] * 100

print("Correct predictions:", correct)
print("Total test images:", Y_test_set.shape[1])
print(f"Test accuracy: {accuracy:.2f}%")

Correct predictions: 16276
Total test images: 21000
Test accuracy: 77.50%


/tmp/ipykernel_26557/3019461778.py:8: RuntimeWarning: overflow encountered in exp
  activated_neuron = 1 / (1 + np.exp(-((weights[layer] @ activated_neuron) + biases[layer])))
